1. 프롬프팅 시뮬레이션 — Zero-shot / Few-shot / CoT / ReAct 비교
2. ReActStep / ReActTrace 클래스 — ReAct 루프 구조 시뮬레이션
3. ReActPromptTemplate 클래스 — 프롬프트 자동 생성기
4. ReActParser 클래스 — 정규표현식 기반 출력 파서

In [2]:
# ─────────────────────────────────────────
# 비교할 질문 하나를 변수로 정의
# 나중에 4가지 기법 모두 이 질문에 적용할 것
# ─────────────────────────────────────────
question = '대한민국의 수도는 어디이며, 그 도시의 인구는 얼마인가?'


# ─────────────────────────────────────────
# [1] Zero-shot: 예시 없이 질문만 던지는 방식
# 가장 단순하지만 복잡한 질문엔 한계가 있음
# ─────────────────────────────────────────
zero_shot_prompt = f'''질문:{question}\n답변:'''


# ─────────────────────────────────────────
# [2] Few-shot: 비슷한 예시를 먼저 보여주고 질문
# 모델이 "아, 이런 형식으로 답하면 되는구나" 학습
# ─────────────────────────────────────────
few_shot_prompt = f'''질문:일본의 수도는 어디이며, 그 도시의 인구는 얼마인가
답변: 일본의 수도는 도쿄이며,인구는 약 1,500만 명입니다.
질문:{question}\n답변:'''


# ─────────────────────────────────────────
# [3] Chain-of-Thought(CoT): 단계별 사고 과정을 명시
# 추론 과정을 드러내서 정확도를 높이는 방법
# 하지만 외부 도구 호출은 여전히 불가
# ─────────────────────────────────────────
cot_prompt = f'''질문:{question}\n단계별로 생각해보겠습니다.\n1단계: 대한민국의 수도를 확인합니다.
2단계: 해당 도시의 인구 데이터를 조회합니다.\n답변:'''


# ─────────────────────────────────────────
# [4] ReAct: 추론(Thought) + 행동(Action) + 관찰(Observation) 반복
# Action으로 외부 도구를 호출하고
# Observation으로 결과를 받아 다시 추론 → 환각 감소
# Finish[...]가 나오면 루프 종료
# ─────────────────────────────────────────
react_prompt = f'''질문:{question}
Thought:이 질문에 답하려면 두 가지 정보가 필요합니다. 먼저 대한민국의 수도를 확인하고, 그 다음 해당도시의 인구를 조회해야합니다.
Action : Search[대한민국수도]
Observation: 대한민국의 수도는 서울특별시입니다.
Thought: 서울이 대한민국의 수도인것을 확인했습니다. 이제 서울의인구를 조회해야 합니다.
Action : Search[서울특별시 인구]
Observation:서울특별시의 인구는 약 900만명입니다.
Thought:두 가지 정보를 모두 확인했으므로 최종 답변을 구성합니다
Action: Finish[대한민국의 수도는 서울이며, 서울의 인구는 약 900만 명입니다.]
'''


# ─────────────────────────────────────────
# 4가지 프롬프트를 순서대로 출력해서 차이를 비교
# ─────────────────────────────────────────
print("=" * 60)
print("[1] Zero-shot Prompting")
print("=" * 60)
print(zero_shot_prompt)

print("=" * 60)
print("[2] Few-shot Prompting")
print("=" * 60)
print(few_shot_prompt)

print("=" * 60)
print("[3] Chain-of-Thought Prompting")
print("=" * 60)
print(cot_prompt)

print("=" * 60)
print("[4] ReAct Prompting")
print("=" * 60)
print(react_prompt)

[1] Zero-shot Prompting
질문:대한민국의 수도는 어디이며, 그 도시의 인구는 얼마인가?
답변:
[2] Few-shot Prompting
질문:일본의 수도는 어디이며, 그 도시의 인구는 얼마인가
답변: 일본의 수도는 도쿄이며,인구는 약 1,500만 명입니다.
질문:대한민국의 수도는 어디이며, 그 도시의 인구는 얼마인가?
답변:
[3] Chain-of-Thought Prompting
질문:대한민국의 수도는 어디이며, 그 도시의 인구는 얼마인가?
단계별로 생각해보겠습니다.
1단계: 대한민국의 수도를 확인합니다.
2단계: 해당 도시의 인구 데이터를 조회합니다.
답변:
[4] ReAct Prompting
질문:대한민국의 수도는 어디이며, 그 도시의 인구는 얼마인가?
Thought:이 질문에 답하려면 두 가지 정보가 필요합니다. 먼저 대한민국의 수도를 확인하고, 그 다음 해당도시의 인구를 조회해야합니다.
Action : Search[대한민국수도]
Observation: 대한민국의 수도는 서울특별시입니다.
Thought: 서울이 대한민국의 수도인것을 확인했습니다. 이제 서울의인구를 조회해야 합니다.
Action : Search[서울특별시 인구]
Observation:서울특별시의 인구는 약 900만명입니다.
Thought:두 가지 정보를 모두 확인했으므로 최종 답변을 구성합니다
Action: Finish[대한민국의 수도는 서울이며, 서울의 인구는 약 900만 명입니다.]



In [3]:
# ─────────────────────────────────────────
# ReAct 루프의 "한 번의 사이클"을 표현하는 클래스
# 한 스텝 = Thought + Action + Action_input + Observation
# ─────────────────────────────────────────
class ReActStep:

    # ─────────────────────────────────────────
    # step_num      : 몇 번째 스텝인지 (1, 2, 3...)
    # thought       : 이 스텝에서 LLM이 한 추론
    # action        : 어떤 도구를 쓸지 (Search, Calculate 등)
    # action_input  : 도구에 넘길 입력값
    # observation   : 도구 실행 후 돌아온 결과
    # ─────────────────────────────────────────
    def __init__(self, step_num, thought, action, action_input, observation):
        self.step_num = step_num
        self.thought = thought
        self.action = action
        self.action_input = action_input
        self.observation = observation

    # ─────────────────────────────────────────
    # 이 스텝의 내용을 보기 좋게 출력하는 메서드
    # Action은 Tool이름[입력값] 형태로 출력
    # ─────────────────────────────────────────
    def display(self):
        print(f'-- step {self.step_num} --')
        print(f'Thought: {self.thought}')
        print(f'Action: {self.action}[{self.action_input}]')
        print(f'Observation: {self.observation}')
        print()


# ─────────────────────────────────────────
# 실제로 스텝 하나 만들어서 출력해보기
# ─────────────────────────────────────────
step = ReActStep(
    step_num=1,
    thought="대한민국의 수도를 먼저 검색해야 합니다.",
    action="Search",
    action_input="대한민국 수도",
    observation="대한민국의 수도는 서울특별시입니다."
)

step.display()

-- step 1 --
Thought: 대한민국의 수도를 먼저 검색해야 합니다.
Action: Search[대한민국 수도]
Observation: 대한민국의 수도는 서울특별시입니다.



In [5]:
# ─────────────────────────────────────────
# ReAct 루프 전체를 관리하는 클래스
# ReActStep들을 리스트로 쌓고
# 최종 답변(final_answer)까지 관리
# ─────────────────────────────────────────
class ReActTrace:

    # ─────────────────────────────────────────
    # question     : 처음에 던진 질문
    # steps        : ReActStep 객체들을 담는 리스트 (빈 리스트로 시작)
    # final_answer : Finish[...] 로 종료될 때 저장되는 최종 답변
    # ─────────────────────────────────────────
    def __init__(self, question):
        self.question = question
        self.steps = []           # 스텝이 쌓일수록 리스트가 늘어남
        self.final_answer = None  # 아직 답변 없으므로 None으로 초기화

    # ─────────────────────────────────────────
    # 스텝을 하나씩 추가하는 메서드
    # ReActStep을 직접 만들어서 steps 리스트에 append
    # step_num은 현재 리스트 길이 + 1로 자동 계산 → 번호 직접 안 넣어도 됨
    # return self → 메서드 체이닝 가능 (add_step().add_step() 형태)
    # ─────────────────────────────────────────
    def add_step(self, thought, action, action_input, observation):
        step = ReActStep(len(self.steps) + 1, thought, action, action_input, observation)
        self.steps.append(step)
        return self  # 체이닝을 위해 자기 자신을 반환

    # ─────────────────────────────────────────
    # 루프를 종료하고 최종 답변을 저장하는 메서드
    # Finish[...] 액션이 나왔을 때 호출
    # return self → 마찬가지로 체이닝 가능
    # ─────────────────────────────────────────
    def finish(self, answer):
        self.final_answer = answer
        return self

    # ─────────────────────────────────────────
    # 전체 루프를 출력하는 메서드
    # 질문 → 각 스텝 → 최종 답변 순서로 출력
    # ─────────────────────────────────────────
    def display(self):
        print("=" * 60)
        print(f"Question: {self.question}")
        print("=" * 60)
        for step in self.steps:   # steps 리스트를 순서대로 순회
            step.display()        # 2단계에서 만든 display() 재사용
        if self.final_answer:     # final_answer가 있을 때만 출력
            print(f"Final Answer: {self.final_answer}")
        print("=" * 60)


# ─────────────────────────────────────────
# 실제 사용 예시
# 2024 노벨 물리학상 수상자를 찾는 다단계 질문
# ─────────────────────────────────────────
trace = ReActTrace("2024년 노벨 물리학상 수상자는 누구이며, 그의 주요 연구 분야는?")

# 스텝 추가 (체이닝으로 연결)
trace.add_step(
    thought="이 질문에 답하려면 2024년 노벨 물리학상 수상자를 먼저 찾아야 합니다.",
    action="Search",
    action_input="2024년 노벨 물리학상 수상자",
    observation="2024년 노벨 물리학상은 존 홉필드(John Hopfield)와 제프리 힌턴(Geoffrey Hinton)이 수상했습니다."
).add_step(
    thought="수상자를 확인했습니다. 이제 이들의 주요 연구 분야를 조회해야 합니다.",
    action="Search",
    action_input="존 홉필드 제프리 힌턴 연구 분야",
    observation="존 홉필드는 홉필드 네트워크를, 제프리 힌턴은 역전파 알고리즘 등 인공 신경망 분야를 개척했습니다."
)

# 루프 종료 + 최종 답변 저장
trace.finish(
    "2024년 노벨 물리학상 수상자는 존 홉필드와 제프리 힌턴이며, "
    "이들의 주요 연구 분야는 인공 신경망과 기계 학습의 기초 이론입니다."
)

trace.display()

Question: 2024년 노벨 물리학상 수상자는 누구이며, 그의 주요 연구 분야는?
-- step 1 --
Thought: 이 질문에 답하려면 2024년 노벨 물리학상 수상자를 먼저 찾아야 합니다.
Action: Search[2024년 노벨 물리학상 수상자]
Observation: 2024년 노벨 물리학상은 존 홉필드(John Hopfield)와 제프리 힌턴(Geoffrey Hinton)이 수상했습니다.

-- step 2 --
Thought: 수상자를 확인했습니다. 이제 이들의 주요 연구 분야를 조회해야 합니다.
Action: Search[존 홉필드 제프리 힌턴 연구 분야]
Observation: 존 홉필드는 홉필드 네트워크를, 제프리 힌턴은 역전파 알고리즘 등 인공 신경망 분야를 개척했습니다.

Final Answer: 2024년 노벨 물리학상 수상자는 존 홉필드와 제프리 힌턴이며, 이들의 주요 연구 분야는 인공 신경망과 기계 학습의 기초 이론입니다.


#### 실제 AI Agent
사용자 질문 입력
      ↓
LLM이 Thought + Action 텍스트를 생성  ← 여기서 AI가 직접 추론
      ↓
Parser가 텍스트에서 action, action_input 추출
      ↓
실제 Search 도구 실행 (진짜 검색 API 호출)
      ↓
결과(Observation)를 다시 LLM에게 전달
      ↓
LLM이 다시 Thought + Action 생성
      ↓
Finish가 나올 때까지 반복

In [6]:
# ─────────────────────────────────────────
# LLM에게 넘길 프롬프트를 조립하는 클래스
# 시스템 지시문 + 도구 목록 + 예시 + 질문
# 이 4가지를 조합해서 완성된 프롬프트를 만듦
# ─────────────────────────────────────────
class ReActPromptTemplate:

    def __init__(self):
        self.tools = []              # 사용 가능한 도구 목록 (리스트)
        self.examples = []           # Few-shot 예시 목록 (리스트)
        self.system_instruction = "" # 에이전트 역할/규칙 정의 (문자열)

    # ─────────────────────────────────────────
    # 도구를 추가하는 메서드
    # name           : 도구 이름 (LLM이 Action에서 쓸 이름)
    # description    : 도구 설명 (LLM이 언제 쓸지 판단하는 근거)
    # usage_example  : 사용법 예시 (Search[검색어] 형태)
    # return self    : 체이닝 가능
    # ─────────────────────────────────────────
    def add_tool(self, name, description, usage_example):
        self.tools.append({
            'name': name,
            'description': description,
            'usage': usage_example
        })
        return self

    # ─────────────────────────────────────────
    # Few-shot 예시를 추가하는 메서드
    # question    : 예시 질문
    # react_trace : Thought→Action→Observation 흐름을 담은 문자열
    # LLM이 이 예시를 보고 출력 형식을 따라함
    # ─────────────────────────────────────────
    def add_example(self, question, react_trace):
        self.examples.append({
            'question': question,
            'react_trace': react_trace
        })
        return self

    # ─────────────────────────────────────────
    # 에이전트의 역할과 규칙을 정의하는 메서드
    # 이게 LLM의 행동 방식 전체를 결정함
    # ─────────────────────────────────────────
    def set_system_instruction(self, instruction):
        self.system_instruction = instruction
        return self

    # ─────────────────────────────────────────
    # 최종 프롬프트를 조립하는 메서드
    # prompt_parts 리스트에 각 섹션을 순서대로 추가하고
    # 마지막에 "\n".join()으로 하나의 문자열로 합침
    # ─────────────────────────────────────────
    def build(self, user_question):
        prompt_parts = []  # 섹션별로 쌓을 리스트

        # [1] 시스템 지시문 추가
        prompt_parts.append(self.system_instruction)
        prompt_parts.append("")  # 빈 줄로 섹션 구분

        # [2] 도구 목록 추가
        # LLM이 이 목록을 보고 어떤 도구를 쓸지 판단
        prompt_parts.append("사용 가능한 도구:")
        for tool in self.tools:
            prompt_parts.append(f"  - {tool['name']}: {tool['description']}")
            prompt_parts.append(f"    사용법: {tool['usage']}")
        prompt_parts.append("")  # 빈 줄로 섹션 구분

        # [3] 출력 형식 명시
        # LLM이 반드시 이 형식으로 출력하도록 강제
        # → 파서(5단계)가 이 형식을 기준으로 파싱하기 때문에 중요
        prompt_parts.append("출력 형식:")
        prompt_parts.append("Thought: [현재 상황에 대한 추론]")
        prompt_parts.append("Action: [도구 이름][입력값]")
        prompt_parts.append("Observation: [도구 실행 결과]")
        prompt_parts.append("... (필요한 만큼 반복)")
        prompt_parts.append("Thought: [최종 추론]")
        prompt_parts.append("Action: Finish[최종 답변]")
        prompt_parts.append("")  # 빈 줄로 섹션 구분

        # [4] Few-shot 예시 추가
        # 예시가 있을 때만 추가 (없으면 Zero-shot)
        if self.examples:
            prompt_parts.append("예시:")
            for i, ex in enumerate(self.examples, 1):  # 1부터 번호 시작
                prompt_parts.append(f"--- 예시 {i} ---")
                prompt_parts.append(f"Question: {ex['question']}")
                prompt_parts.append(ex['react_trace'])
                prompt_parts.append("")

        # [5] 실제 유저 질문 추가 (항상 마지막)
        prompt_parts.append(f"Question: {user_question}")

        # 리스트를 줄바꿈으로 연결해서 하나의 문자열로 반환
        return "\n".join(prompt_parts)


# ─────────────────────────────────────────
# 실제 사용 예시
# ─────────────────────────────────────────

# 템플릿 인스턴스 생성
template = ReActPromptTemplate()

# 시스템 지시문 설정
template.set_system_instruction(
    "당신은 주어진 질문에 정확하게 답변하기 위해 도구를 사용하는 AI 에이전트입니다\n"
    "반드시 Thought->Action->Observation 순서를 따르며, 최종 답변은 Finish 액션으로 제출합니다"
)

# 도구 3개 등록
template.add_tool('Search', '위키피디아에서 정보를 검색합니다.', 'Search[검색어]')
template.add_tool('Lookup', '현재 문서에서 특정 키워드를 찾습니다.', 'Lookup[키워드]')
template.add_tool('Calculator', '수학 계산을 수행합니다.', 'Calculate[수식]')

# Few-shot 예시 추가 (LLM이 형식을 따라하도록)
example_trace = """Thought: 프랑스의 수도를 먼저 검색해야 합니다.
Action: Search[프랑스 수도]
Observation: 프랑스의 수도는 파리(Paris)입니다.
Thought: 파리의 인구를 추가로 검색해야 합니다.
Action: Search[파리 인구]
Observation: 파리의 인구는 약 210만 명입니다(2023년 기준).
Thought: 필요한 정보를 모두 확보했습니다.
Action: Finish[프랑스의 수도는 파리이며, 인구는 약 210만 명입니다.]"""

template.add_example("프랑스의 수도는 어디이며, 그 도시의 인구는 얼마인가?", example_trace)

# 최종 프롬프트 생성
prompt = template.build("독일의 수도는 어디이며, 면적은 얼마인가?")
print(prompt)

당신은 주어진 질문에 정확하게 답변하기 위해 도구를 사용하는 AI 에이전트입니다
반드시 Thought->Action->Observation 순서를 따르며, 최종 답변은 Finish 액션으로 제출합니다

사용 가능한 도구:
  - Search: 위키피디아에서 정보를 검색합니다.
    사용법: Search[검색어]
  - Lookup: 현재 문서에서 특정 키워드를 찾습니다.
    사용법: Lookup[키워드]
  - Calculator: 수학 계산을 수행합니다.
    사용법: Calculate[수식]

출력 형식:
Thought: [현재 상황에 대한 추론]
Action: [도구 이름][입력값]
Observation: [도구 실행 결과]
... (필요한 만큼 반복)
Thought: [최종 추론]
Action: Finish[최종 답변]

예시:
--- 예시 1 ---
Question: 프랑스의 수도는 어디이며, 그 도시의 인구는 얼마인가?
Thought: 프랑스의 수도를 먼저 검색해야 합니다.
Action: Search[프랑스 수도]
Observation: 프랑스의 수도는 파리(Paris)입니다.
Thought: 파리의 인구를 추가로 검색해야 합니다.
Action: Search[파리 인구]
Observation: 파리의 인구는 약 210만 명입니다(2023년 기준).
Thought: 필요한 정보를 모두 확보했습니다.
Action: Finish[프랑스의 수도는 파리이며, 인구는 약 210만 명입니다.]

Question: 독일의 수도는 어디이며, 면적은 얼마인가?


In [7]:
# ─────────────────────────────────────────
# 정규표현식을 사용하기 위한 모듈
# LLM이 출력한 비정형 텍스트에서
# Thought, Action, Observation을 추출할 때 사용
# ─────────────────────────────────────────
import re
from typing import List, Dict, Optional, Tuple
# List     : 리스트 타입 힌트
# Dict     : 딕셔너리 타입 힌트
# Optional : None이 될 수도 있는 타입 (Optional[str] = str 또는 None)
# Tuple    : 튜플 타입 힌트 (예: Tuple[str, str] = ("Search", "검색어"))


class ReActParser:

    # ─────────────────────────────────────────
    # 클래스 변수로 패턴을 미리 컴파일해서 저장
    # 인스턴스 생성 없이 클래스 자체에서 바로 사용 가능
    # re.compile() : 패턴을 미리 컴파일 → 반복 사용시 속도 빠름
    # re.DOTALL    : .이 줄바꿈(\n)도 포함해서 매칭되도록 설정
    # ─────────────────────────────────────────

    # "Thought: ..." 추출
    # (?=\nAction|$) : Action이 나오거나 문자열 끝까지만 매칭
    #               → Thought 영역을 넘어서 Action까지 잡지 않도록 경계 설정
    THOUGHT_PATTERN = re.compile(r'Thought\s*:\s*(.+?)(?=\nAction|$)', re.DOTALL)

    # "Action: ToolName[input]" 추출
    # (\w+)  : 도구 이름 (Search, Calculate 등 영문+숫자)
    # (.+?)  : 도구 입력값 (대괄호 안의 내용)
    ACTION_PATTERN = re.compile(r'Action\s*:\s*(\w+)\[(.+?)\]', re.DOTALL)

    # "Observation: ..." 추출
    # (?=\nThought|$) : 다음 Thought가 나오거나 끝까지만 매칭
    OBSERVATION_PATTERN = re.compile(r'Observation\s*:\s*(.+?)(?=\nThought|$)', re.DOTALL)

    # "Action: Finish[최종답변]" 추출
    # 루프 종료 신호를 감지할 때 사용
    FINISH_PATTERN = re.compile(r'Action\s*:\s*Finish\[(.+?)\]', re.DOTALL)


    # ─────────────────────────────────────────
    # @classmethod : self 대신 cls를 받음
    # 인스턴스 없이 ReActParser.parse_single_step() 으로 바로 호출 가능
    # 텍스트 한 블록에서 Thought/Action/Observation을 한번에 추출
    # ─────────────────────────────────────────
    @classmethod
    def parse_single_step(cls, text: str) -> Dict:

        # 결과를 담을 딕셔너리, 매칭 안되면 None 유지
        result = {
            "thought": None,
            "action": None,
            "action_input": None,
            "observation": None
        }

        # Thought 추출
        # .search() : 텍스트 전체에서 패턴 찾기 (첫번째 매칭만)
        # .group(1) : 괄호()로 감싼 첫번째 캡처 그룹 반환
        thought_match = cls.THOUGHT_PATTERN.search(text)
        if thought_match:
            result["thought"] = thought_match.group(1).strip()  # 앞뒤 공백 제거

        # Action 이름 + 입력값 추출
        # group(1) = 도구 이름, group(2) = 입력값
        action_match = cls.ACTION_PATTERN.search(text)
        if action_match:
            result["action"] = action_match.group(1).strip()
            result["action_input"] = action_match.group(2).strip()

        # Observation 추출
        obs_match = cls.OBSERVATION_PATTERN.search(text)
        if obs_match:
            result["observation"] = obs_match.group(1).strip()

        return result


    # ─────────────────────────────────────────
    # LLM이 출력한 전체 텍스트를 파싱하는 메서드
    # Thought 키워드를 기준으로 텍스트를 스텝별로 분리한 뒤
    # 각 스텝을 parse_single_step()으로 파싱
    # ─────────────────────────────────────────
    @classmethod
    def parse_full_trace(cls, text: str) -> Dict:
        steps = []
        final_answer = None

        # Finish 패턴이 있으면 최종 답변 먼저 추출
        finish_match = cls.FINISH_PATTERN.search(text)
        if finish_match:
            final_answer = finish_match.group(1).strip()

        # "Thought:" 키워드 앞에서 텍스트를 분리
        # (?=Thought\s*:) : Thought: 앞에서 분리 (Thought: 자체는 유지)
        thought_splits = re.split(r'(?=Thought\s*:)', text)
        thought_splits = [s.strip() for s in thought_splits if s.strip()]  # 빈 문자열 제거

        # 각 블록을 parse_single_step()으로 파싱
        for split in thought_splits:
            step = cls.parse_single_step(split)
            if step["thought"]:  # thought가 있는 유효한 스텝만 추가
                steps.append(step)

        return {
            "steps": steps,
            "final_answer": final_answer,
            "num_steps": len(steps)  # 총 스텝 수
        }


    # ─────────────────────────────────────────
    # Finish 패턴이 있는지만 확인 → True/False 반환
    # ReAct 루프를 멈출 타이밍을 감지할 때 사용
    # ─────────────────────────────────────────
    @classmethod
    def is_finished(cls, text: str) -> bool:
        return bool(cls.FINISH_PATTERN.search(text))


    # ─────────────────────────────────────────
    # Action에서 (도구이름, 입력값) 튜플만 빠르게 추출
    # ToolRegistry에 바로 넘길 때 사용
    # Optional[Tuple[str,str]] : 매칭 실패시 None 반환 가능
    # ─────────────────────────────────────────
    @classmethod
    def extract_action(cls, text: str) -> Optional[Tuple[str, str]]:
        match = cls.ACTION_PATTERN.search(text)
        if match:
            return (match.group(1).strip(), match.group(2).strip())
        return None  # 매칭 안되면 None 반환


# ─────────────────────────────────────────
# 실제 사용 예시
# ─────────────────────────────────────────
test_trace = """Thought: 양자 컴퓨팅의 기본 원리를 먼저 조사해야 합니다.
Action: Search[양자 컴퓨팅 기본 원리]
Observation: 양자 컴퓨팅은 큐비트(qubit)를 사용하며, 중첩과 얽힘의 원리를 활용합니다.
Thought: 기본 원리를 확인했습니다. 이제 기존 컴퓨팅과의 차이점을 조사해야 합니다.
Action: Search[양자 컴퓨팅 기존 컴퓨팅 차이]
Observation: 기존 컴퓨팅은 비트(0 또는 1)를 사용하지만, 양자 컴퓨팅은 동시에 여러 상태를 표현할 수 있습니다.
Thought: 충분한 정보를 확보했으므로 최종 답변을 작성합니다.
Action: Finish[양자 컴퓨팅은 큐비트를 기반으로 중첩과 얽힘을 활용하며, 기존 컴퓨팅보다 특정 문제에서 기하급수적으로 빠른 연산이 가능합니다.]"""

# 전체 파싱
result = ReActParser.parse_full_trace(test_trace)
print("=== 파싱 결과 ===")
print(f"총 단계 수: {result['num_steps']}")
for i, step in enumerate(result['steps'], 1):
    print(f"\n--- Step {i} ---")
    print(f"  Thought    : {step['thought']}")
    print(f"  Action     : {step['action']}")
    print(f"  Input      : {step['action_input']}")
    print(f"  Observation: {step['observation']}")
print(f"\n최종 답변: {result['final_answer']}")
print(f"종료 여부: {ReActParser.is_finished(test_trace)}")

# extract_action 단독 사용 예시
# → ToolRegistry에 바로 넘길 때 이렇게 씀
action_tuple = ReActParser.extract_action("Action: Search[양자 컴퓨팅]")
print(f"\nextract_action 결과: {action_tuple}")
# 출력 → ('Search', '양자 컴퓨팅')

=== 파싱 결과 ===
총 단계 수: 3

--- Step 1 ---
  Thought    : 양자 컴퓨팅의 기본 원리를 먼저 조사해야 합니다.
  Action     : Search
  Input      : 양자 컴퓨팅 기본 원리
  Observation: 양자 컴퓨팅은 큐비트(qubit)를 사용하며, 중첩과 얽힘의 원리를 활용합니다.

--- Step 2 ---
  Thought    : 기본 원리를 확인했습니다. 이제 기존 컴퓨팅과의 차이점을 조사해야 합니다.
  Action     : Search
  Input      : 양자 컴퓨팅 기존 컴퓨팅 차이
  Observation: 기존 컴퓨팅은 비트(0 또는 1)를 사용하지만, 양자 컴퓨팅은 동시에 여러 상태를 표현할 수 있습니다.

--- Step 3 ---
  Thought    : 충분한 정보를 확보했으므로 최종 답변을 작성합니다.
  Action     : Finish
  Input      : 양자 컴퓨팅은 큐비트를 기반으로 중첩과 얽힘을 활용하며, 기존 컴퓨팅보다 특정 문제에서 기하급수적으로 빠른 연산이 가능합니다.
  Observation: None

최종 답변: 양자 컴퓨팅은 큐비트를 기반으로 중첩과 얽힘을 활용하며, 기존 컴퓨팅보다 특정 문제에서 기하급수적으로 빠른 연산이 가능합니다.
종료 여부: True

extract_action 결과: ('Search', '양자 컴퓨팅')
